# Aula 04 · Bisseção e falsa posição

Esta aula apresenta o [capítulo 4 do site](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/). A ideia central: **toda pergunta "qual x faz a conta dar tal resultado?" vira f(x) = 0**, e uma raiz presa num intervalo onde f troca de sinal pode ser encurralada até a precisão que se quiser.

**Ao fim da aula você consegue:**

1. transformar um problema numa equação $f(x) = 0$ e localizar a raiz num gráfico;
2. deduzir e aplicar a bisseção, inclusive numa tabela à mão, e prever quantas iterações ela precisa;
3. deduzir e aplicar a falsa posição, e reconhecer quando ela emperra;
4. conferir o resultado com o `brentq` do `scipy`.

**Roteiro:** 🧩 · 1. o gráfico · 2. 🧑‍🏫 bisseção · 3. quantas iterações · 4. 🧑‍🏫 falsa posição · 5. quando emperra · 6. confira · 7. outra área · 🎯 prática · 🧩 o remédio · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Farmácia hospitalar — a janela terapêutica.**
>
> *Depois de um comprimido, a concentração de um antibiótico no sangue sobe, chega
> a um pico e cai. Ele só faz efeito acima de **5 mg/L**. O médico receitou "de 8 em
> 8 horas", e a farmacêutica desconfia: "**por quanto tempo cada comprimido fica
> acima da dose eficaz? O paciente fica algum tempo desprotegido entre uma dose e
> outra?**"*

As duas pontas da janela — quando a concentração **passa** de 5 mg/L e quando
**volta** abaixo — são raízes de $C(t) - 5 = 0$. No fim da aula, você as calcula.

## 1. Achar uma raiz é achar onde a curva cruza o zero

Que massa faz um paraquedista chegar a 36 m/s depois de 4 s? A fórmula de $v$ não
se deixa isolar para $m$, mas $f(m) = v(m) - 36$ vale **zero** na massa procurada.
O primeiro passo é sempre olhar o gráfico.

📖 [capítulo 4 · Achar uma raiz é achar onde a curva cruza o zero](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#achar-uma-raiz-e-achar-onde-a-curva-cruza-o-zero)

In [ ]:
# 📦 dados prontos — só rode esta célula
# O paraquedista de trás para frente: f(m) = 0 na massa que leva a 36 m/s em 4 s.
g = 9.81
c = 0.25
t = 4


def f(m):
    v = np.sqrt(g * m / c) * np.tanh(np.sqrt(g * c / m) * t)
    return v - 36

> 🧰 **Comando novo: `plt.axhline`**
>
> `plt.axhline(y)` desenha uma linha horizontal atravessando a figura inteira, na
> altura `y`. Com `plt.axhline(0, color="black")`, o zero fica marcado e dá para ver
> onde a curva o cruza.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
x = [-2, -1, 0, 1, 2, 3]
y = [-5, -2, -1, 0, 3, 8]
plt.figure()
plt.plot(x, y, "o-")
plt.axhline(0, color="black")
plt.grid()
plt.show()

**✍️ Passo 1.** Desenhe `f` para `massas = np.linspace(50, 200, 100)`, com `plt.axhline(0, color="black")`, nomes nos eixos e `plt.show()`.

In [ ]:
# ✍️ passo 1

**Preveja:** entre que massas a curva cruza o zero?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Entre 100 e 200 kg, perto de 140. À esquerda da raiz, $f < 0$ (o paraquedista
leve ainda não chegou a 36 m/s); à direita, $f > 0$.

</details>

**✍️ Passo 2.** Imprima `f(100)`, `f(200)` e o produto `f(100) * f(200)`.

In [ ]:
# ✍️ passo 2

**Preveja:** qual o sinal do produto? O que ele garante?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Negativo ($-1{,}20 \times 0{,}86$). Produto negativo quer dizer **sinais
contrários**: como $f$ não dá saltos, ela passou pelo zero entre 100 e 200.

📖 [capítulo 4 · Achar uma raiz é achar onde a curva cruza o zero](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#achar-uma-raiz-e-achar-onde-a-curva-cruza-o-zero)

</details>

## 2. Bisseção

📖 [capítulo 4 · Bisseção](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#bissecao)

### 🧑‍🏫 No quadro — o método da bisseção

Caderno de papel aberto. No quadro:

1. $f$ contínua e $f(a)\,f(b) < 0$: existe raiz em $[a, b]$ (Bolzano);
2. o ponto médio $x_m = (a + b)/2$;
3. o sinal de $f(a)\,f(x_m)$ escolhe a metade que ainda tem a raiz;
4. repetir até $\varepsilon_a$ ficar abaixo da tolerância;
5. uma tabela à mão com $a$, $b$, $x_m$, $f(x_m)$ e $\varepsilon_a$.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

A cada iteração o intervalo cai **pela metade**, e a raiz continua presa nele.
Depois de $n$ iterações, o erro é no máximo $\dfrac{b - a}{2^n}$.

| it | $a$ | $b$ | $x_m$ | $f(x_m)$ |
|---|---|---|---|---|
| 1 | 100 | 200 | 150 | +0,142 |
| 2 | 100 | 150 | 125 | −0,409 |
| 3 | 125 | 150 | 137,5 | −0,111 |

</details>

**✍️ Passo 3.** Com `a = 100.0` e `b = 200.0`, calcule `xm`, imprima `f(xm)` e decida: a raiz está em `[a, xm]` ou em `[xm, b]`? Atualize `a` ou `b`.

In [ ]:
# ✍️ passo 3

**Preveja:** depois do primeiro corte, qual é o novo intervalo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$x_m = 150$, $f(150) = +0{,}142$. Como $f(100) < 0$, o produto $f(a)\,f(x_m)$ é
negativo, e a raiz está em $[100, 150]$: faz-se `b = xm`.

</details>

**✍️ Passo 4.** Agora num laço: recomece com `a = 100.0`, `b = 200.0` e repita o corte 11 vezes, imprimindo `a`, `b` e `xm` a cada volta.

In [ ]:
# ✍️ passo 4

**Preveja:** com 11 cortes, quantos quilos de largura sobram no intervalo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$100 / 2^{11} \approx 0{,}05$ kg. O ponto médio final é 142,72 kg, e a raiz
verdadeira (142,74) está a menos de 50 g dele.

</details>

> ⚠️ **Armadilha.** Trocar o lado errado (`a = xm` onde devia ser `b = xm`) não dá erro: o método
continua cortando ao meio e converge para uma das **pontas**, que não é raiz.
Confira sempre se $f$ no resultado está perto de zero.

### 🎯 Sua vez — Um passo da bisseção

Escreva `um_passo(f, a, b)`, que faz **um** corte da bisseção e devolve a
tupla `(a, b)` do novo intervalo.

In [ ]:
def um_passo(f, a, b):
    # sua solução aqui
    pass

In [ ]:
def cubo_menos_20(x):
    return x**3 - 20


confere(um_passo, [
    ((cubo_menos_20, 2, 3), (2.5, 3)),
    ((cubo_menos_20, 2.5, 3), (2.5, 2.75)),
])

<details>
<summary><b>💡 Dica</b></summary>

Ponto médio, teste `f(a) * f(xm) < 0` e dois `return` diferentes.

</details>

## 3. Quantas iterações são necessárias

A bisseção é a única que avisa **antes** quanto vai trabalhar: para o erro ficar
abaixo de $\Delta$, basta $n > \log_2\big((b-a)/\Delta\big) = \ln\big((b-a)/\Delta\big) / \ln 2$.

📖 [capítulo 4 · Quantas iterações são necessárias](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#quantas-iteracoes-sao-necessarias)

**✍️ Passo 5.** Calcule `np.log((200 - 100) / 0.01) / np.log(2)`.

In [ ]:
# ✍️ passo 5

**Preveja:** quantas iterações garantem a massa com erro menor que 10 gramas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$13{,}3$, então **14 iterações** — para essa função ou para qualquer outra, com
o mesmo intervalo inicial. É a garantia que os outros métodos não dão.

</details>

## 4. Falsa posição

📖 [capítulo 4 · Falsa posição](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#falsa-posicao)

### 🧑‍🏫 No quadro — a falsa posição

Caderno de papel aberto. No quadro:

1. se $|f(b)| \ll |f(a)|$, a raiz deve estar mais perto de $b$;
2. a reta de $(a, f(a))$ a $(b, f(b))$ e onde ela cruza o zero (semelhança de
   triângulos);
3. o resto igual à bisseção: teste de sinal e troca de um dos lados.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$ x_r = b - \frac{f(b)\,(a - b)}{f(a) - f(b)} $$

</details>

**✍️ Passo 6.** Repita o laço do passo 4 trocando o ponto médio por `xr = b - f(b) * (a - b) / (f(a) - f(b))`. Faça 6 voltas e imprima `a`, `b` e `xr`.

In [ ]:
# ✍️ passo 6

**Preveja:** a falsa posição chega mais perto em menos voltas? Algum dos extremos fica parado?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Em 6 voltas, 142,760 kg — a bisseção precisou de 11. E o `a` **nunca sai de
100**: a curva é côncava, e a reta cruza o zero sempre do mesmo lado da raiz.

📖 [capítulo 4 · Falsa posição](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#falsa-posicao)

</details>

### 🎯 Sua vez — O ponto da falsa posição

Escreva `ponto_falsa_posicao(f, a, b)`, que devolve o ponto $x_r$ onde a reta entre $(a, f(a))$ e $(b, f(b))$ cruza o zero.

In [ ]:
def ponto_falsa_posicao(f, a, b):
    # sua solução aqui
    pass

In [ ]:
confere(ponto_falsa_posicao, [
    ((cubo_menos_20, 2, 3), 2.6315789473684212),
    ((np.sin, 3, 3.3), 3.1416556068691603),
])

<details>
<summary><b>💡 Dica</b></summary>

Uma linha só: a fórmula do quadro. Cuidado com os parênteses.

</details>

## 5. Quando a falsa posição emperra

Em $f(x) = x^{10} - 1$, a curva é quase plana perto de zero e sobe como um paredão
perto da raiz ($x = 1$). A célula 📦 roda os dois métodos por 30 iterações e guarda
o erro de cada uma.

📖 [capítulo 4 · Quando a falsa posição emperra](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#quando-a-falsa-posicao-emperra)

In [ ]:
# 📦 dados prontos — só rode esta célula
# f(x) = x**10 - 1 em [0, 1.3]: o erro (distância até a raiz, 1) em 30 iterações
def f10(x):
    return x**10 - 1


a, b = 0.0, 1.3
erros_bis = []
for it in range(30):
    xm = (a + b) / 2
    erros_bis.append(abs(xm - 1))
    if f10(a) * f10(xm) < 0:
        b = xm
    else:
        a = xm

a, b = 0.0, 1.3
erros_fp = []
for it in range(30):
    xr = b - f10(b) * (a - b) / (f10(a) - f10(b))
    erros_fp.append(abs(xr - 1))
    if f10(a) * f10(xr) < 0:
        b = xr
    else:
        a = xr
print("erros calculados")

> 🧰 **Comando novo: `plt.semilogy`**
>
> `plt.semilogy(x, y)` funciona como o `plt.plot`, mas só o eixo **vertical** fica em
> escala logarítmica. É o gráfico certo para "erro × iteração": um erro que cai 10
> vezes por iteração vira uma **reta** descendo.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
plt.figure()
plt.semilogy([1, 2, 3, 4, 5], [0.1, 0.01, 0.001, 0.0001, 0.00001], "o-")
plt.grid()
plt.show()

**✍️ Passo 7.** Desenhe `erros_bis` e `erros_fp` contra `range(1, 31)` com `plt.semilogy`, com `label=` e `plt.legend()`.

In [ ]:
# ✍️ passo 7

**Preveja:** qual dos dois chega mais perto da raiz em 30 iterações?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A bisseção desce em reta até $10^{-10}$; a falsa posição quase não sai de
$10^{-3}$. A reta dela cruza o zero sempre longe da raiz e um extremo fica
preso. **Método mais esperto nem sempre é método mais rápido.**

📖 [capítulo 4 · Quando a falsa posição emperra](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#quando-a-falsa-posicao-emperra)

</details>

## 6. Confira com a biblioteca

📖 [capítulo 4 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#confira-com-a-biblioteca)

> 🧰 **Comando novo: `from scipy.optimize import brentq`**
>
> O `scipy` é uma biblioteca de métodos numéricos prontos. `brentq(f, a, b)` acha uma
> raiz de `f` entre `a` e `b` (onde `f` troca de sinal), misturando bisseção com
> métodos mais rápidos. No curso, o `scipy` serve para **conferir** o que você
> escreveu, nunca para substituir.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
from scipy.optimize import brentq

print(brentq(np.sin, 3, 3.3))    # a raiz de sen(x) perto de 3 é pi

**✍️ Passo 8.** Importe o `brentq` e calcule `brentq(f, 100, 200)`.

In [ ]:
# ✍️ passo 8

**Preveja:** os seus resultados da bisseção e da falsa posição estão dentro de 0,05 % dele?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`142.7376`. A bisseção deu 142,7246 e a falsa posição, 142,7598: ambas a menos
de 0,02 % da resposta, dentro da tolerância.

📖 [capítulo 4 · Confira com a biblioteca](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#confira-com-a-biblioteca)

</details>

## 7. Mesmo método, outra área

**Finanças.** Investir R\$ 10 000 hoje e receber R\$ 3 000 por ano durante 5 anos
rende quanto ao ano? É a **taxa interna de retorno** (TIR): a taxa $r$ que zera o
valor presente líquido, $\text{VPL}(r) = -10\,000 + \sum_{k=1}^{5} \dfrac{3000}{(1+r)^k}$.

📖 [capítulo 4 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#mesmo-metodo-outra-area)

**✍️ Passo 9.** Escreva `vpl(r)` (um laço de 1 a 5 somando `3000 / (1 + r)**ano`, começando em `-10000`), imprima `vpl(0.10)` e `vpl(0.20)` e depois `brentq(vpl, 0.10, 0.20)`.

In [ ]:
# ✍️ passo 9

**Preveja:** a TIR fica mais perto de 10 % ou de 20 %?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

VPL(10 %) = +1372 e VPL(20 %) = −1028: a TIR é **15,24 %** ao ano, um pouco mais
perto de 20 %. O mesmo `brentq` que achou a massa do paraquedista achou os
juros: o método não sabe de que área é o problema.

📖 [capítulo 4 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

Retoma o bloco *7. Mesmo método, outra área*.
📖 [capítulo 4 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/04-bissecao-falsa-posicao/#mesmo-metodo-outra-area)

### 🎯 Sua vez — O VPL de qualquer investimento

Escreva `vpl(r, fluxos)`, em que `fluxos[k]` é o dinheiro que entra (positivo)
ou sai (negativo) no ano `k` — `fluxos[0]` é hoje, sem desconto. Devolva a
soma de `fluxos[k] / (1 + r)**k`.

In [ ]:
def vpl(r, fluxos):
    # sua solução aqui
    pass

In [ ]:
confere(vpl, [
    ((0.1, [-10000, 3000, 3000, 3000, 3000, 3000]), 1372.3603082253417),
    ((0.0, [-500, 200, 200, 200]), 100.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Acumulador com `for ano in range(len(fluxos)):`. No ano 0, `(1 + r)**0` vale 1.

</details>

## 🧩 Resolvendo o problema

> *"**Por quanto tempo cada comprimido fica acima da dose eficaz? O paciente fica
> algum tempo desprotegido entre uma dose e outra?**"* — a farmacêutica.

A célula 📦 define a concentração `C(t)`, a dose eficaz e a função `excesso(t)`,
que vale zero nas duas pontas da janela. O gráfico mostra onde elas estão.

In [ ]:
# 📦 dados prontos — só rode esta célula
# Concentração do remédio no sangue (mg/L), t horas depois do comprimido.
def C(t):
    return 20 * (np.exp(-0.2 * t) - np.exp(-1.5 * t))


DOSE_EFICAZ = 5.0     # abaixo disso o remédio não faz efeito (mg/L)


def excesso(t):
    return C(t) - DOSE_EFICAZ


ts = np.linspace(0, 12, 200)
plt.figure()
plt.plot(ts, C(ts))
plt.axhline(DOSE_EFICAZ, color="black")
plt.xlabel("horas depois do comprimido")
plt.ylabel("concentração (mg/L)")
plt.grid()
plt.show()

### 🎯 Sua vez — A bisseção como função

Escreva `bissecao(f, a, b, tol)`, que aplica a bisseção a `f` em `[a, b]` e
para quando $\varepsilon_a$ (em %) ficar abaixo de `tol` (no máximo 100
iterações). Devolva o último ponto médio.

In [ ]:
def bissecao(f, a, b, tol):
    # sua solução aqui
    pass

In [ ]:
confere(bissecao, [
    ((cubo_menos_20, 2, 3, 1e-4), 2.714418411254883),
], tol=1e-5)

<details>
<summary><b>💡 Dica</b></summary>

O laço do passo 4, com o `if` do seu 🎯 `um_passo` e o $\varepsilon_a$ do
capítulo 1: guarde o ponto médio anterior antes de calcular o novo.

</details>

Agora a janela terapêutica. Olhando o gráfico, a primeira raiz está entre 0 e 1,5 h (antes do pico) e a segunda, entre 1,5 e 12 h:

In [ ]:
inicio = bissecao(excesso, 0.01, 1.5, 0.001)
fim = bissecao(excesso, 1.5, 12, 0.001)
print("início do efeito:", inicio, "h")
print("fim do efeito:   ", fim, "h")

<details>
<summary><b>▶ O que os números dizem</b></summary>

O remédio passa de 5 mg/L **14 minutos** depois do comprimido e volta
abaixo **6.93 h** depois: são **6.7 horas** de efeito. Com doses de 8
em 8 horas, sobra cerca de **1 hora** entre uma janela e a próxima em que o
paciente fica abaixo da dose eficaz.

A farmacêutica tem um argumento para sugerir ao médico um intervalo de **6 em 6
horas** — ou uma dose maior. (Na prática, a segunda dose se soma ao que sobrou da
primeira, e a conta real é um pouco mais favorável; mas o método é o mesmo.)

</details>

## 📋 A lista

Abra a [Lista 04](https://lacouth.github.io/metodos_telecom-site/listas/lista04/). O **Exercício 01** é à mão (✏️): a bisseção de
$x^3 - 20$ em $[2, 3]$. Comece por ele, no papel.

**a)** Qual o primeiro ponto médio, e qual o sinal de $f$ nele?

<details>
<summary><b>▶ Resposta</b></summary>

$x_m = 2{,}5$ e $f(2{,}5) = 15{,}625 - 20 = -4{,}375$: negativo, como $f(2)$. A raiz está em $[2{,}5;\ 3]$.

</details>

**b)** Quantas iterações garantem erro menor que 0,001?

<details>
<summary><b>▶ Resposta</b></summary>

$\log_2(1/0{,}001) = 9{,}97$: **10 iterações**.

</details>

Termine a tabela e siga para o **Exercício 02**, a bisseção como função.

## 🚪 Antes de sair

**1.** Por que a bisseção precisa de $f(a)$ e $f(b)$ com sinais contrários?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque é essa troca de sinal que **garante** (para $f$ contínua) que existe uma raiz no intervalo. Sem ela, o método corta ao meio do mesmo jeito e converge para um ponto qualquer.

</details>

**2.** A função $f(x) = (x - 2)^2$ tem raiz em $x = 2$. A bisseção consegue achá-la? Por quê?

<details>
<summary><b>▶ Resposta da 2</b></summary>

**Não**: $(x-2)^2$ nunca é negativa, então não há intervalo com troca de sinal. Raízes em que a curva só **toca** o zero escapam dos métodos de intervalo — o próximo capítulo tem métodos que não precisam de troca de sinal.

</details>

**3.** Por que a falsa posição, mesmo "mais esperta", pode ser mais lenta que a bisseção?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Porque em funções muito curvas a reta cruza o zero sempre do mesmo lado da raiz, e um dos extremos fica parado: o intervalo encolhe só de um lado, devagar.

</details>

## 🏠 Para casa

- Refaça no papel 4 iterações da bisseção de $x^3 - 20$ **sem olhar**.
- Termine a [Lista 04](https://lacouth.github.io/metodos_telecom-site/listas/lista04/).
- Leia o começo do [capítulo 5](https://lacouth.github.io/metodos_telecom-site/unidade3-raizes/05-newton-secante/): e se, em
  vez de uma reta entre dois pontos, a gente usasse a **tangente**?